# TenantAid - AI-Powered Tenant Rights Triage
**SDG 1: No Poverty** | Alameda County, CA

---

TenantAid: a two-step AI pipeline that helps Alameda County renters understand eviction notices and know what to do next.

**Step 1 - Structured Extraction (Lab 2 capability):**  
The AI reads the tenant's notice or description and extracts a fixed schema: notice type, reason, deadline, amount owed, unit type, and language. This mirrors Lab 2's structured extraction work.

**Step 2 - Plain-Language Generation (Lab 1 capability):**  
Using the extracted schema, the AI generates a plain-language summary explaining what the notice means, which protections apply, and what the tenant must do in the tenant's own language.


## Setup

In [14]:
# Install the Gemini SDK
!pip install -q google-genai

import google.generativeai as genai
import json

from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

genai.configure(api_key=GEMINI_API_KEY)

# We use gemini-2.5-flash - same model as the labs, free-tier friendly
model = genai.GenerativeModel("gemini-2.5-flash")
print("Gemini configured. Ready to run TenantAid.")

Gemini configured. Ready to run TenantAid.


---
## 1. The Extraction Schema

This is the core of Step 1. We define exactly what fields we want the AI to fill in, just like Lab 2's structured extraction, but adapted for eviction notices instead of 311 complaints.

**Design decision:** We constrain the output to JSON with a fixed schema so the same fields appear every time. Without this constraint (as we saw in Lab 2 without a schema), the model produces different output shapes on every run - which makes it impossible to match the extracted data against legal rules automatically.

In [15]:
# --- EXTRACTION SCHEMA ---
# These are the fields we need to determine which Alameda County tenant
# protections apply. Each field maps to a legal rule.
#
# Fields:
#   notice_type      : type of legal notice received
#   reason           : stated reason for the notice
#   amount_owed      : dollar amount if pay-or-quit notice
#   notice_date      : date on the notice
#   response_deadline: how many days the tenant has to respond
#   unit_type        : helps determine which protections apply (AB 1482 exempts some)
#   language         : language of the original notice or tenant's message
#   urgency          : HIGH / MEDIUM / LOW - HIGH triggers human review
#   ambiguous        : true if the reason or notice type is unclear - triggers human review

EXTRACTION_SYSTEM_PROMPT = """
You are a legal document parser for a tenant rights tool in Alameda County, California.

Your job is ONLY to extract structured information from eviction notices or tenant descriptions.
You do NOT give legal advice. You do NOT interpret ambiguous language in the landlord's favor
or the tenant's favor - you flag it as ambiguous.

Respond ONLY with a valid JSON object. No preamble, no explanation, no markdown fences.

Use this exact schema:
{
  "notice_type": "3-day-pay-or-quit | 3-day-cure-or-quit | 3-day-quit | 30-day-notice |
                  60-day-notice | unlawful-detainer | lockout | other | unknown",
  "reason": "nonpayment | lease-violation-curable | lease-violation-incurable |
             nuisance | owner-move-in | ellis-act | no-fault | other | unknown",
  "amount_owed": "dollar amount as string, or null if not applicable",
  "notice_date": "date as string, or null if not found",
  "response_deadline_days": "integer, or null if not found",
  "unit_type": "single-family | condo | apartment | mobile-home | unknown",
  "language": "english | spanish | vietnamese | cantonese | tagalog | other",
  "urgency": "HIGH | MEDIUM | LOW",
  "ambiguous": true or false
}

Set ambiguous=true if: the reason is unclear, the notice type is mixed or contradictory,
or the text contains legal language that doesn't fit a clear category.
Set urgency=HIGH if: notice_type is 3-day-quit, 3-day-pay-or-quit, unlawful-detainer, or lockout.
"""

def extract_notice_fields(tenant_input: str) -> dict:
    """
    Step 1: Extract structured fields from a tenant's notice or description.
    Returns a Python dict of the extracted schema.

    Design note: We send the system prompt + tenant input as a single user message
    because Gemini's generateContent handles system instructions via system_instruction param.
    The key constraint is JSON-only output - no fences, no prose.
    """
    extraction_model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=EXTRACTION_SYSTEM_PROMPT
    )
    response = extraction_model.generate_content(tenant_input)
    raw = response.text.strip()

    # Strip markdown fences if the model adds them despite instructions
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]

    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        # If parsing fails, return a safe fallback that triggers human review
        return {
            "notice_type": "unknown",
            "reason": "unknown",
            "amount_owed": None,
            "notice_date": None,
            "response_deadline_days": None,
            "unit_type": "unknown",
            "language": "english",
            "urgency": "HIGH",
            "ambiguous": True
        }

print("Extraction function ready.")

Extraction function ready.


---
## 2. The Generation Prompt

Step 2 takes the extracted schema and generates a plain-language explanation for the tenant.

**Key design decisions:**
- The system prompt explicitly says the model is NOT a lawyer and should NOT give legal advice, it explains options, not conclusions
- The prompt includes Alameda County-specific context (AB 1482, just-cause ordinance) so the model doesn't hallucinate generic national rules
- Language detection from Step 1 is passed in - if the tenant wrote in Spanish, the response comes back in Spanish
- Ambiguous cases get a different message: "we need a human to review this before giving you next steps"

In [16]:
# Alameda County legal context we inject into every generation call.
# This is the key differentiator from a generic chatbot - the model
# knows the specific local protections without us having to fine-tune it.
ALAMEDA_CONTEXT = """
ALAMEDA COUNTY TENANT PROTECTIONS (as of 2025):
- AB 1482 (statewide): Just-cause eviction protections for most rentals built before 2005,
  rented for 12+ months. Landlords must have a valid reason to evict.
- Fremont does NOT have a local rent control ordinance, but AB 1482 caps rent increases
  at 5% + CPI for covered units.
- A "3-day notice to quit" (with no option to cure) for lease violations may be legally
  defective in many cases - many violations require a "3-day cure or quit" first.
- Tenants have the right to request a reasonable repayment plan for COVID-era rent debt.
- Key local resources: Legal Aid of Alameda County (510-250-5270),
  Bay Area Legal Aid, Alameda County Rent Assistance Program.
"""

GENERATION_SYSTEM_PROMPT = """
You are TenantAid, a plain-language tenant rights information tool for Alameda County, California.

You are NOT a lawyer. You do NOT give legal advice. You explain what a notice means,
what options exist under Alameda County and California law, and what the tenant should do next.
Always end by recommending the tenant contact a local legal aid organization.

Write clearly at a 6th-grade reading level. No legal jargon without explanation.
If the tenant's language is Spanish, respond entirely in Spanish.
If the tenant's language is Vietnamese, respond entirely in Vietnamese.
Otherwise, respond in English.

If the notice is flagged as ambiguous or HIGH urgency, open with a clear warning:
"IMPORTANT: This notice needs a legal review before you take any action."

Structure your response in these sections:
1. What this notice is saying (1-2 sentences)
2. Whether this notice appears valid (explain simply - do not give a legal conclusion)
3. Which Alameda County protections may apply to you
4. What you should do in the next 24-72 hours (numbered steps)
5. Who to call first (specific organization and phone number)
"""

def generate_tenant_summary(extracted: dict, original_input: str) -> str:
    """
    Step 2: Generate a plain-language action summary for the tenant.
    Takes the extracted schema dict and the original input text.
    Returns a plain-language string in the tenant's language.
    """
    # Build a structured prompt combining the extracted data + local context
    prompt = f"""
{ALAMEDA_CONTEXT}

Here is what we extracted from the tenant's notice:
{json.dumps(extracted, indent=2)}

Here is the tenant's original message:
\"\"\"{original_input}\"\"\"

Tenant's language: {extracted.get('language', 'english')}
Urgency level: {extracted.get('urgency', 'MEDIUM')}
Ambiguous: {extracted.get('ambiguous', False)}

Write the plain-language summary for this tenant.
"""
    generation_model = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=GENERATION_SYSTEM_PROMPT
    )
    response = generation_model.generate_content(prompt)
    return response.text

print("Generation function ready.")

Generation function ready.


---
## 3. Full Pipeline Function + Human Review Flag

This ties Steps 1 and 2 together and implements the oversight decision:
**HIGH urgency or ambiguous cases are flagged for human review.**

In [17]:
def run_tenantaid(tenant_input: str, verbose: bool = True):
    """
    Full TenantAid pipeline:
    1. Extract structured fields from tenant input
    2. Check for human review trigger (HIGH urgency or ambiguous)
    3. Generate plain-language summary
    4. Return both the structured record and the tenant-facing summary
    """
    print("=" * 60)
    print("TENANTAID - Alameda County Tenant Rights Triage")
    print("=" * 60)
    print(f"\nInput received:\n{tenant_input}\n")

    # STEP 1: Extract structured fields
    print("[Step 1] Extracting structured fields...")
    extracted = extract_notice_fields(tenant_input)

    if verbose:
        print("\n--- Extracted Schema (JSON) ---")
        print(json.dumps(extracted, indent=2))

    # OVERSIGHT CHECK: Flag cases that need human review before the tenant
    # receives a response. This is our Part 4 oversight decision.
    # Justification: In Lab 2, we saw the model resolve ambiguous inputs confidently.
    # For 3-day notices and lockouts, a wrong answer causes irreversible harm.
    needs_human_review = (
        extracted.get("urgency") == "HIGH" or
        extracted.get("ambiguous") == True or
        extracted.get("notice_type") in ["unlawful-detainer", "lockout", "unknown"]
    )

    if needs_human_review:
        print("\n" + "!" * 60)
        print("HUMAN REVIEW REQUIRED before sending to tenant.")
        print("Reason: HIGH urgency or ambiguous notice type detected.")
        print("Action: Route to Legal Aid intake coordinator within 24 hours.")
        print("!" * 60)

    # STEP 2: Generate plain-language summary
    print("\n[Step 2] Generating plain-language tenant summary...")
    summary = generate_tenant_summary(extracted, tenant_input)

    print("\n--- Tenant-Facing Summary ---")
    print(summary)

    print("\n--- Caseworker Record (for intake system) ---")
    caseworker_record = {
        "extracted_fields": extracted,
        "needs_human_review": needs_human_review,
        "original_input": tenant_input
    }
    print(json.dumps(caseworker_record, indent=2))

    return extracted, summary, needs_human_review

print("Pipeline ready. Run the test cases below.")

Pipeline ready. Run the test cases below.


---
## 4. Test Case A - Rosa's 3-Day Notice (the core use case)

This is the scenario from our README: Rosa receives a 3-day pay-or-quit notice and doesn't know what to do.

In [18]:
# Test Case A: Standard 3-day pay-or-quit notice
# Expected: HIGH urgency, nonpayment reason, valid notice type
# Expected generation: Spanish response (because input is in Spanish)

rosa_input = """
Me pusieron un papel en la puerta que dice "3-Day Notice to Pay Rent or Quit".
Dice que debo $1,850 de renta de noviembre. Mi landlord es una compania, no una persona.
Llevo 3 años viviendo aqui en Fremont en un apartamento. No se que hacer.
"""

extracted_a, summary_a, review_a = run_tenantaid(rosa_input)

TENANTAID - Alameda County Tenant Rights Triage

Input received:

Me pusieron un papel en la puerta que dice "3-Day Notice to Pay Rent or Quit".
Dice que debo $1,850 de renta de noviembre. Mi landlord es una compania, no una persona.
Llevo 3 años viviendo aqui en Fremont en un apartamento. No se que hacer.


[Step 1] Extracting structured fields...

--- Extracted Schema (JSON) ---
{
  "notice_type": "3-day-pay-or-quit",
  "reason": "nonpayment",
  "amount_owed": "$1,850",
  "notice_date": null,
  "response_deadline_days": 3,
  "unit_type": "apartment",
  "language": "english",
  "urgency": "HIGH",
  "ambiguous": false
}

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
HUMAN REVIEW REQUIRED before sending to tenant.
Reason: HIGH urgency or ambiguous notice type detected.
Action: Route to Legal Aid intake coordinator within 24 hours.
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!

[Step 2] Generating plain-language tenant summary...

--- Tenant-Facing Summary -

---
## 5. Test Case B - The Failure Case (ambiguous notice)

This is the failure case from our Part 4 / README. The notice says "3-day to vacate" for a "lease violation" but doesn't say what the violation is and doesn't give an option to cure.

**What we expect to observe:** The model should flag this as ambiguous=true because it cannot determine whether this should be a "cure or quit" (which the tenant can respond to) or a valid "quit only" notice (which has stricter criteria). If it fills in the fields confidently without flagging ambiguity, that IS the failure case we documented.

In [19]:
# Test Case B: Ambiguous violation notice - our documented failure case
# Expected: ambiguous=True, urgency=HIGH, triggers human review flag
# Watch for: Does the model flag the ambiguity, or resolve it confidently?
# This output is what we reference in Part 4 of the assignment.

ambiguous_input = """
I got a notice taped to my door that says:
"NOTICE TO QUIT - You are hereby required to vacate the premises within 3 days
due to lease violations. Failure to comply will result in legal action."
It doesn't say what the violation is. It doesn't give me a chance to fix anything.
I've lived here 2 years in an apartment in Fremont. I don't know if this is real.
"""

extracted_b, summary_b, review_b = run_tenantaid(ambiguous_input)

TENANTAID - Alameda County Tenant Rights Triage

Input received:

I got a notice taped to my door that says:
"NOTICE TO QUIT - You are hereby required to vacate the premises within 3 days
due to lease violations. Failure to comply will result in legal action."
It doesn't say what the violation is. It doesn't give me a chance to fix anything.
I've lived here 2 years in an apartment in Fremont. I don't know if this is real.


[Step 1] Extracting structured fields...

--- Extracted Schema (JSON) ---
{
  "notice_type": "3-day-quit",
  "reason": "lease-violation-incurable",
  "amount_owed": null,
  "notice_date": null,
  "response_deadline_days": 3,
  "unit_type": "apartment",
  "language": "english",
  "urgency": "HIGH",
  "ambiguous": true
}

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
HUMAN REVIEW REQUIRED before sending to tenant.
Reason: HIGH urgency or ambiguous notice type detected.
Action: Route to Legal Aid intake coordinator within 24 hours.
!!!!!!!!!!!!!!!!!!!!!!

---
## 6. Test Case C - Lower urgency (30-day notice, no-fault)

This tests a different notice type: a 30-day no-fault notice. AB 1482 requires just cause for these in covered units, so the AI should flag whether this notice appears valid depending on how long the tenant has lived there.

In [20]:
# Test Case C: 30-day notice with no stated reason
# Expected: MEDIUM urgency, no-fault reason, may trigger AB 1482 just-cause check

thirty_day_input = """
My landlord gave me a 30-day notice to vacate with no reason given.
I have been renting this apartment for 4 years. The building has about 20 units.
Built in the 1990s I think. Fremont CA. Is this legal?
"""

extracted_c, summary_c, review_c = run_tenantaid(thirty_day_input)

TENANTAID - Alameda County Tenant Rights Triage

Input received:

My landlord gave me a 30-day notice to vacate with no reason given.
I have been renting this apartment for 4 years. The building has about 20 units.
Built in the 1990s I think. Fremont CA. Is this legal?


[Step 1] Extracting structured fields...

--- Extracted Schema (JSON) ---
{
  "notice_type": "30-day-notice",
  "reason": "no-fault",
  "amount_owed": null,
  "notice_date": null,
  "response_deadline_days": 30,
  "unit_type": "apartment",
  "language": "english",
  "urgency": "LOW",
  "ambiguous": false
}

[Step 2] Generating plain-language tenant summary...

--- Tenant-Facing Summary ---
Here is information about your 30-day notice:

### 1. What this notice is saying
Your landlord has given you a notice that says you must move out of your apartment in 30 days. The notice does not give a reason why you need to leave.

### 2. Whether this notice appears valid
Based on what you've told us (you've lived there for 4 years

---
## 7. Reflection - What did the outputs show?


In [21]:
# Summary comparison of all three test cases
# This is what we reference in our Part 4 failure case analysis

cases = [
    ("A - Rosa's 3-Day Pay-or-Quit",   extracted_a, review_a),
    ("B - Ambiguous Violation Notice",  extracted_b, review_b),
    ("C - 30-Day No-Fault Notice",      extracted_c, review_c),
]

print("\n" + "=" * 60)
print("COMPARISON: All Three Test Cases")
print("=" * 60)

for name, ext, review in cases:
    print(f"\n{name}")
    print(f"  notice_type      : {ext.get('notice_type')}")
    print(f"  reason           : {ext.get('reason')}")
    print(f"  urgency          : {ext.get('urgency')}")
    print(f"  ambiguous        : {ext.get('ambiguous')}")
    print(f"  language         : {ext.get('language')}")
    print(f"  human review?    : {'YES - routed to coordinator' if review else 'No - sent to tenant'}")

print("\n" + "=" * 60)
print("Lab 2 connection:")
print("In Lab 2, the model routed the homeless encampment complaint to")
print("Environmental Services, a confident output on an ambiguous input.")
print("Test Case B tests whether TenantAid makes the same kind of error.")
print("If ambiguous=False on Case B, that IS the failure we documented.")
print("=" * 60)


COMPARISON: All Three Test Cases

A - Rosa's 3-Day Pay-or-Quit
  notice_type      : 3-day-pay-or-quit
  reason           : nonpayment
  urgency          : HIGH
  ambiguous        : False
  language         : english
  human review?    : YES - routed to coordinator

B - Ambiguous Violation Notice
  notice_type      : 3-day-quit
  reason           : lease-violation-incurable
  urgency          : HIGH
  ambiguous        : True
  language         : english
  human review?    : YES - routed to coordinator

C - 30-Day No-Fault Notice
  notice_type      : 30-day-notice
  reason           : no-fault
  urgency          : LOW
  ambiguous        : False
  language         : english
  human review?    : No - sent to tenant

Lab 2 connection:
In Lab 2, the model routed the homeless encampment complaint to
Environmental Services, a confident output on an ambiguous input.
Test Case B tests whether TenantAid makes the same kind of error.
If ambiguous=False on Case B, that IS the failure we documented

---
## Notes on AI Assistance

- **Gemini 2.5 Flash** was used for all API calls (same model as labs)
- The `extract_notice_fields` function structure was AI-assisted; we modified the schema fields and added the fallback error handling
- The Alameda County legal context block was researched and written by the team; AI was used to check phrasing
- All test case inputs were written by the team based on real notice types; outputs are from actual API calls

---
